<a href="https://colab.research.google.com/github/oumuchiha007/Jirakit-ponkan/blob/main/Class_Hotel%2C_Loop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###



In [ ]:
class Hotel:

  def __init__(self, name, rooms, guests, bookings, rejected_log):
    self.name = name
    self.rooms = []
    self.guests = {}
    self.bookings = []
    self.rejected_log = []

  def add_room(self, room):
    self.rooms.append(room)
  def add_guest(self, guest):
    self.guests[guest.guest_id] = guest
  def find_available_room(self, room_type, check_in, nights):
    candidates = [
        r for r in self.rooms
        if r.room_type == room_type
        and r.is_available(check_in, nights)
    ]

    if not candidates:
      return None
    return random.choice(candidates)

    def occupancy_on(self, day):
          occupied = sum(
              1 for r in self.rooms
              if dat in r.booked_dates
          )
          return occupied / len(self.rooms)

    def create_booking(
        self,
        guest,
        room_type,
        check_in,
        nights,
        channel,
        booking_date=None,
        breakfast=False,
        adults=2,
        children=0,
        rate_multiplier=1.0
    ):
        upgraded = False
        room = self.find_available_room(
            room_type,
            check_in,
            nights
        )

        if room is None and room_type in upgraded_path:
          room = self.find_available_room(
              upgrade_path[room_type],
              check_in,
              nights
          )
          if room is not None:
            upgraded = True

        if room is None:
          self.rejected_log.append({
              "guest_id": guest.guest_id,
              "room_type": room_type,
              "check_in": check_in.isoformat(),
              "nights": nights,
              "reason": "ห้องเต็มทุกประเภทที่รองรับได้"
          })

          return None

        booking = Booking(
            len(self.bookings) + 1,
            guest,
            room,
            check_in,
            nights,
            channel,
            booking_date or check_in,
            breakfast,
            upgraded,
            adults,
            children,
            rate_multiplier

        )

        room.reserve(check_in, nights)

        guest.record_stay()

        self.bookings.append(booking)

        return booking

    def occupancy_rate(self, start, end):

          total_nights = lem(self.room) * (
              (end - start).days + 1
          )

          sold_nights = sum(
              1
              for r in self.rooms
              for d in r.booked_dates
              if start <= d <= end
          )

          return round(
              sold_nights / total_nights * 100,
              2
          )

    def adr(self, start, end):
          revenue = 0.0
          nights = 0

          for b in self.bookings:
            if b.is_revenue() and start <= b.check_in <= end:
              revenue += b.price_detail["room_charge"]
              nights += b.nights

          return round(
              revenue / nights,
              2
          ) if nights else 0.0

    def revpar(self, start, end):
      return round(
          self.adr(start, end)
          * self.occupancy_rate(start, end)
          / 100,
          2
      )

    def __repr__(self):
      return (
            f"Hotel({self.name}, "
            f"{len(self.rooms)} ห้อง, "
            f"{len(self.bookings)} ใบจอง)"
        )


###LOOP

In [ ]:
from urllib import request
import random

from src.booking import Booking
from src.config import UPGRADE_PATH

import random
from datetime import timedelta

from src.config import(
    BOOKING_WINDOW_DAYS,
    DAILY_REQUEST_RATE,
    SIM_END,
    SIM_START
)

from src.guest import guest
from src.helpers import (
    daterange,
    damand_factor,
    dynamic_rate_miltiplier,
    generate_guest_name,
    generate_phone,
    pick_channel,
    pick_nationality,
    pick_room_type,
    pick_travel_party,
    poisson,
    random_lead_time,
    random_nights,
    wants_breakfast
)
from src.profiles import(
    CANCEL_RATE_BY_CHANNEL,
    NO_SHOW_RATE
)

def create_walk_in_guest(guest_id):
    nat = pick_nationality()

    return guest(
        guest_id,
        generate_guest_name(nat),
        generate_phone(nat),
        nat
    )

def run_simulation(
    hotel,
    target_bookings=None,
    repeat_rate=0.22,
    verbose=False
):

    next_guest_id = 1
    requests = 0

    booking_start = (
        SIM_START - timedelta(day=  BOOKING_WINDOW_DAYS)
    )

    for today in daterange(
        booking_start,
        SIM_END
    ):

        n_requests = poisson(
            DAILY_REQUEST_RATE
        )

        for _ in range(n_requests):

          if (
              target_bookings
              and len(hotel.booking) >= target_bookings
          ):

              return summarize(
                    hotel,
                    requests
              )

          request += 1

          is_returning  = (
              bool(hotel.guest)
              and random.random() < repeat_rate
          )

          if is_returning:
            guest = random.choice(
                list(hotel.guest.value())
            )

          else:

            guess = create_walk_in_guest(
                next_guest_id
            )

          nat = guest.nationality

          lead = (
              0
              if random.random() < 0.07
              else random_lead_time(nat)
          )

          check_in = (
              today + timedelta(days=lead)
          )

          if (
              check_in >  SIM_END
              or check_in < SIM_START
          ):
              continue

          if random.random() > min(
              demand_factor(check_in) / 1.9,
              1.0
          ):
              continue

          party_kind, adults, children = (
              pick_travel_party()
          )

          room_type = pick_room_type(
              party_kind
          )

          nights = random_nights(nat)

          channel = pick_channel(
              nat,
              lead
          )

          breakfast = wants_breakfast(
              nat
          )

          occ = hotel.occupancy_on(
              check_in
          )

          rate_mult = dynamic_rate_miltiplier(
              occ
          )

          booking = hotel.create_booking(
              guest,
              room_type,
              check_in,
              nights,
              channel,
              booking_date=today,
              breakfast=breakfast,
              adults=adults,
              children=children,
              rate_multiplier=rate_mult
          )

          if booking is None:
            continue
          if not is_returning:

            hotel.add_guest(guest)
            next_guest_id += 1

          if verbose:
            print(booking)

          cancel_prob = (
                CANCEL_RATE_BY_CHANNEL[channel]
          )

          if lead > 30:

                cancel_prob *= 1.25

          elif lead <= 3:

                cancel_prob *= 0.4

          if random.random() < cancel_prob:

                days_before = random.randint(
                    0,
                    max(lead, 1)
                )

                booking.cancel(
                    check_in
                    - timedelta(days=days_before)
                )

          elif random.random() < NO_SHOW_RATE:

                booking.mark_no_show()

          elif booking.check_out <= SIM_END:

                booking.check_out_guest()

    return summarize(
        hotel,
        requests
    )


def summarize(hotel, requests):

    counts = {}

    for b in hotel.bookings:

        counts[b.status] = (
            counts.get(b.status, 0) + 1
        )

    return {
        "requests": requests,
        "bookings": len(hotel.bookings),
        "rejected": len(hotel.rejected_log),
        "guests": len(hotel.guests),
        "status_counts": counts,
        "occupancy": hotel.occupancy_rate(
            SIM_START,
            SIM_END
        ),
        "adr": hotel.adr(
            SIM_START,
            SIM_END
        ),
        "revpar": hotel.revpar(
            SIM_START,
            SIM_END
        ),
    }


ModuleNotFoundError: No module named 'src'